In [10]:
from kiteconnect import KiteConnect
import pandas as pd
import datetime as dt
import os
import numpy as np

In [3]:
request_token=open("request_token.txt","r").read()
key_secrets=open("api_key.txt","r").read().split()
kite = KiteConnect(api_key=key_secrets[0])
data = kite.generate_session(request_token=request_token,api_secret=key_secrets[1])

In [4]:
# Get dump of all NSE instruments
instruments_dump=kite.instruments("NSE")
instruments_df=pd.DataFrame(instruments_dump)
instruments_df.to_csv("NSE_Instruments.csv",index=False)

In [5]:
def instrumentLookup(instruments_df,symbol):
    """Looks up instrument token for a given script from instrument dump"""
    try:
        return instruments_df[instruments_df.tradingsymbol==symbol].instrument_token.values[0]
    except:
        return -1


def fetchOHLC(ticker,interval,duration):
    """extracts historical data and outputs in the form of dataframe"""
    instrument = instrumentLookup(instruments_df,ticker)
    data = pd.DataFrame(kite.historical_data(instrument,dt.date.today()-dt.timedelta(duration), dt.date.today(),interval))
    data.set_index("date",inplace=True)
    return data

In [6]:

def atr(DF,n):
    "function to calculate True Range and Average True Range"
    df = DF.copy()
    df['H-L']=abs(df['high']-df['low'])
    df['H-PC']=abs(df['high']-df['close'].shift(1))
    df['L-PC']=abs(df['low']-df['close'].shift(1))
    df['TR']=df[['H-L','H-PC','L-PC']].max(axis=1,skipna=False)
    df['ATR'] = df['TR'].ewm(com=n,min_periods=n).mean()
    return df['ATR']


In [7]:

def supertrend(DF,n,m):
    """function to calculate Supertrend given historical candle data
        n = n day ATR - usually 7 day ATR is used
        m = multiplier - usually 2 or 3 is used"""
    df = DF.copy()
    df['ATR'] = atr(df,n)
    df["B-U"]=((df['high']+df['low'])/2) + m*df['ATR'] 
    df["B-L"]=((df['high']+df['low'])/2) - m*df['ATR']
    df["U-B"]=df["B-U"]
    df["L-B"]=df["B-L"]
    ind = df.index
    for i in range(n,len(df)):
        if df['close'][i-1]<=df['U-B'][i-1]:
            df.loc[ind[i],'U-B']=min(df['B-U'][i],df['U-B'][i-1])
        else:
            df.loc[ind[i],'U-B']=df['B-U'][i]    
    for i in range(n,len(df)):
        if df['close'][i-1]>=df['L-B'][i-1]:
            df.loc[ind[i],'L-B']=max(df['B-L'][i],df['L-B'][i-1])
        else:
            df.loc[ind[i],'L-B']=df['B-L'][i]  
    df['Strend']=np.nan
    for test in range(n,len(df)):
        if df['close'][test-1]<=df['U-B'][test-1] and df['close'][test]>df['U-B'][test]:
            df.loc[ind[test],'Strend']=df['L-B'][test]
            break
        if df['close'][test-1]>=df['L-B'][test-1] and df['close'][test]<df['L-B'][test]:
            df.loc[ind[test],'Strend']=df['U-B'][test]
            break
    for i in range(test+1,len(df)):
        if df['Strend'][i-1]==df['U-B'][i-1] and df['close'][i]<=df['U-B'][i]:
            df.loc[ind[i],'Strend']=df['U-B'][i]
        elif  df['Strend'][i-1]==df['U-B'][i-1] and df['close'][i]>=df['U-B'][i]:
            df.loc[ind[i],'Strend']=df['L-B'][i]
        elif df['Strend'][i-1]==df['L-B'][i-1] and df['close'][i]>=df['L-B'][i]:
            df.loc[ind[i],'Strend']=df['L-B'][i]
        elif df['Strend'][i-1]==df['L-B'][i-1] and df['close'][i]<=df['L-B'][i]:
            df.loc[ind[i],'Strend']=df['U-B'][i]
    return df['Strend']

In [11]:
ohlc = fetchOHLC("INFY","5minute",5)
strend = supertrend(ohlc,7,3)

In [12]:
ohlc

,open,high,low,close,volume
date,,,,,
2025-04-02 09:15:00+05:30,1535.20,1553.80,1535.20,1551.95,309964
2025-04-02 09:20:00+05:30,1551.95,1552.00,1542.45,1542.45,213776
2025-04-02 09:25:00+05:30,1542.75,1549.85,1542.10,1548.50,144103
2025-04-02 09:30:00+05:30,1548.60,1553.50,1547.45,1553.35,189656
2025-04-02 09:35:00+05:30,1553.35,1553.35,1546.80,1548.80,75944
...,...,...,...,...,...
2025-04-04 15:05:00+05:30,1450.40,1453.55,1449.00,1453.50,248967
2025-04-04 15:10:00+05:30,1453.10,1453.10,1450.60,1451.20,212260
2025-04-04 15:15:00+05:30,1451.25,1453.65,1450.25,1452.45,268750


In [13]:
strend

date
2025-04-02 09:15:00+05:30            NaN
2025-04-02 09:20:00+05:30            NaN
2025-04-02 09:25:00+05:30            NaN
2025-04-02 09:30:00+05:30            NaN
2025-04-02 09:35:00+05:30            NaN
                                ...     
2025-04-04 15:05:00+05:30    1456.787769
2025-04-04 15:10:00+05:30    1456.787769
2025-04-04 15:15:00+05:30    1456.787769
2025-04-04 15:20:00+05:30    1456.787769
2025-04-04 15:25:00+05:30    1456.787769
Name: Strend, Length: 225, dtype: float64